# 🎙️ Bengali ASR & Speaker Diarization - Kaggle Deployment

Run the web app on Kaggle with GPU. Uses ngrok for public access.

**Before running:**
1. Enable GPU in Kaggle (Settings → Accelerator → GPU T4 x2)
2. Add your model datasets as inputs
3. Set your ngrok auth token (free at https://ngrok.com)

In [ ]:
#%% Install dependencies
!apt-get install -q -y ffmpeg
!pip install -q fastapi uvicorn[standard] python-multipart aiofiles pyngrok
!pip install -q faster-whisper noisereduce bnunicodenormalizer silero-vad

# Install DiariZen
import os
if not os.path.exists('/kaggle/working/DiariZen'):
    !git clone https://github.com/BUTSpeechFIT/DiariZen.git /kaggle/working/DiariZen
!pip install -q -e /kaggle/working/DiariZen/pyannote-audio
!pip install -q -e /kaggle/working/DiariZen

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print('✅ Ready!')


In [ ]:
#%% Write web app files
import os

APP_DIR = '/kaggle/working/webapp'
os.makedirs(f'{APP_DIR}/static', exist_ok=True)
os.makedirs(f'{APP_DIR}/uploads', exist_ok=True)
os.makedirs(f'{APP_DIR}/results', exist_ok=True)

# Set model paths - adjust these to match your Kaggle input datasets
os.environ['ASR_MODEL_PATH'] = '/kaggle/input/whisper-md-lora-ep7-ct2/whisper_md_lora_e7_ct2'  # Adjust!
os.environ['DIARIZEN_REPO_ID'] = 'BUT-FIT/diarizen-wavlm-large-s80-md-v2'
os.environ['DIARIZEN_FINETUNED_PATH'] = '/kaggle/input/finetune-segmentation/best_diarizen_retrained.pt'  # Adjust!
os.environ['HF_TOKEN'] = ''  # Your HF token
os.environ['ASR_COMPUTE_TYPE'] = 'float16'

print('✅ Config set')
print(f"  ASR: {os.environ['ASR_MODEL_PATH']}")
print(f"  Diarization fine-tuned: {os.environ['DIARIZEN_FINETUNED_PATH']}")

In [ ]:
#%% Copy webapp files from Kaggle dataset
import shutil, re

SRC_DIR = '/kaggle/input/datasets/tanzirmannanturzo/bangla-webapp'

shutil.copy(f'{SRC_DIR}/config.py',   f'{APP_DIR}/config.py')
shutil.copy(f'{SRC_DIR}/pipeline.py', f'{APP_DIR}/pipeline.py')
shutil.copy(f'{SRC_DIR}/app.py',      f'{APP_DIR}/app.py')
shutil.copy(f'{SRC_DIR}/index.html',  f'{APP_DIR}/static/index.html')

# Hotfix: remove condition_on_prev_text (not supported by BatchedInferencePipeline)
pipeline_path = f'{APP_DIR}/pipeline.py'
with open(pipeline_path, 'r') as f:
    code = f.read()

# Use regex to handle any whitespace variation
patched = re.sub(r'[ \t]*condition_on_prev_text\s*=\s*\S+,?\s*\n', '', code)

if 'condition_on_prev_text' in patched:
    print('❌ Hotfix FAILED - still found condition_on_prev_text in pipeline.py')
    # Show context around the bad line for debugging
    for i, line in enumerate(patched.splitlines(), 1):
        if 'condition_on_prev_text' in line:
            print(f'  Line {i}: {repr(line)}')
else:
    with open(pipeline_path, 'w') as f:
        f.write(patched)
    print('✅ Hotfix applied - condition_on_prev_text removed')

# Verify files exist
for f in ['config.py', 'pipeline.py', 'app.py', 'static/index.html']:
    path = os.path.join(APP_DIR, f)
    status = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'  {status} {f}')


In [ ]:
#%% Debug: verify pipeline.py transcribe() call
pipeline_path = f'{APP_DIR}/pipeline.py'
with open(pipeline_path, 'r') as f:
    lines = f.readlines()

print('--- transcribe() call in deployed pipeline.py ---')
in_block = False
for i, line in enumerate(lines, 1):
    if 'model.transcribe(' in line:
        in_block = True
    if in_block:
        print(f'{i:4d}: {line}', end='')
        if in_block and line.strip() == ')':
            break

if 'condition_on_prev_text' in ''.join(lines):
    print('\n❌ WARNING: condition_on_prev_text still present!')
else:
    print('\n✅ condition_on_prev_text is NOT present - pipeline.py looks good')


In [ ]:
#%% Start server with ngrok
from kaggle_secrets import UserSecretsClient
NGROK_AUTH_TOKEN = UserSecretsClient().get_secret("NGROK_TOKEN")

NGROK_DOMAIN = 'vikki-woody-plicately.ngrok-free.dev'

import sys, threading, time, os
from pyngrok import ngrok, conf

# Write a clean ngrok config (no saved domains) to avoid ERR_NGROK_15013
ngrok_config_path = '/tmp/ngrok_clean.yml'
with open(ngrok_config_path, 'w') as _f:
    _f.write(f'version: "3"\nagent:\n  authtoken: {NGROK_AUTH_TOKEN}\n')

# Kill any existing ngrok, then configure with the clean config
ngrok.kill()
pyngrok_conf = conf.PyngrokConfig(config_path=ngrok_config_path)

os.chdir(APP_DIR)
sys.path.insert(0, APP_DIR)

def run_server():
    import uvicorn
    uvicorn.run('app:app', host='0.0.0.0', port=7860)

threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)

# Connect using your static free domain
tunnel = ngrok.connect(addr=7860, proto="http", domain=NGROK_DOMAIN, pyngrok_config=pyngrok_conf)
public_url = tunnel.public_url

print(f'\n{"="*60}')
print(f'🌐 App live at: {public_url}')
print(f'{"="*60}\n')

try:
    while True: time.sleep(1)
except KeyboardInterrupt:
    ngrok.disconnect(public_url, pyngrok_config=pyngrok_conf)
    ngrok.kill(pyngrok_config=pyngrok_conf)
